Running a Nilearn Demo (oractice for real data)

If a brain map pops up for the cell below, then we are ready for the real thing. 

In [2]:
from nilearn.datasets import fetch_spm_auditory
from nilearn.glm.first_level import FirstLevelModel, make_first_level_design_matrix
from nilearn.plotting import plot_stat_map, show
import pandas as pd
import numpy as np

# Load example auditory dataset
data = fetch_spm_auditory()
fmri_img = data.func[0]
tr = 7.0  # TR of demo dataset

frame_times = np.arange(fmri_img.shape[-1]) * tr

# Simple design: onsets they give us
events = pd.read_table(data['events'])

design_matrix = make_first_level_design_matrix(
    frame_times,
    events=events,
    hrf_model='glover'
)

model = FirstLevelModel(t_r=tr)
model = model.fit(fmri_img, design_matrices=design_matrix)

z_map = model.compute_contrast('active > rest', output_type='z_score')

plot_stat_map(z_map, title='Demo: active > rest')
show()


[wrapper] Added README.md to C:\Users\Cecilia\nilearn_data

[wrapper] Dataset created in C:\Users\Cecilia\nilearn_data\spm_auditory

[wrapper] Data absent, downloading...

[wrapper] Downloading data from https://www.fil.ion.ucl.ac.uk/spm/download/data/MoAEpilot/MoAEpilot.bids.zip ...

[wrapper] Downloaded 1302528 of 30176409 bytes (4.3%%,   25.3s remaining)

...

[wrapper] Downloaded 28999680 of 30176409 bytes (96.1%%,    0.3s remaining)

[wrapper]  ...done. (9 seconds, 0 min)

[wrapper] Extracting data from C:\Users\Cecilia\nilearn_data\spm_auditory\MoAEpilot.bids.zip...

[wrapper] .. done.

AttributeError: 'str' object has no attribute 'shape'

**needs coodination with Behavioral Lead (Theo)

While we wait for preprocessed BOLD, we can:
- decide how Theo should format regressors for us (e.g., one .tsv per run with columns: onset, duration, arousal, music_on)
- agree on file names and folder structure. Ex: 

regressors/
  sub-01/
    run-1_events.tsv
    run-2_events.tsv

- decide whether:
    - arousal will be continuous (parametric)
    - We'll also get a high vs low arousal flag later for PPI

We don't need BOLD data to write code that parses events and builds a design matrix, we can mock it now. 

**Starter Nilearn script template for Neural Cadence** 

Skeleton we can fill in once we have:
- fMRIPrep outputs exists
- Behavioral Lead gives event/regressor files

In [ ]:
"""
Starter GLM script for ds002725 using Nilearn
- Single subject, single run example
- Parametric modulation with arousal
"""

import os
import numpy as np
import pandas as pd

from nilearn import image, plotting
from nilearn.glm.first_level import FirstLevelModel, make_first_level_design_matrix

# ========= 1. PATHS & SUBJECT/RUN CHOICES =========

# TODO: change this to your actual project root on your machine or inside Neurodesk
PROJECT_ROOT = "/path/to/neurodesktop-storage/project"

SUBJECT = "sub-01"
RUN = "run-1"

# fMRIPrep preprocessed BOLD (example filename pattern)
fmri_path = os.path.join(
    PROJECT_ROOT,
    "fmriprep",
    SUBJECT,
    "func",
    f"{SUBJECT}_task-music_{RUN}_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz",
)

# Brain mask from fMRIPrep (or you can let Nilearn infer one)
mask_path = os.path.join(
    PROJECT_ROOT,
    "fmriprep",
    SUBJECT,
    "func",
    f"{SUBJECT}_task-music_{RUN}_space-MNI152NLin2009cAsym_desc-brain_mask.nii.gz",
)

# Confounds from fMRIPrep
confounds_path = os.path.join(
    PROJECT_ROOT,
    "fmriprep",
    SUBJECT,
    "func",
    f"{SUBJECT}_task-music_{RUN}_desc-confounds_timeseries.tsv",
)

# Behavioral/regressor file prepared by Behavioral Lead
# Example: a TSV with columns: onset, duration, trial_type, arousal
events_path = os.path.join(
    PROJECT_ROOT,
    "regressors",
    SUBJECT,
    f"{SUBJECT}_task-music_{RUN}_events.tsv",
)

# ========= 2. LOAD DATA =========

print("Loading fMRI image...")
fmri_img = image.load_img(fmri_path)

print("Loading mask...")
mask_img = image.load_img(mask_path)

print("Loading confounds...")
all_confounds = pd.read_table(confounds_path)

# You can pick a subset of confounds (e.g., motion + aCompCor)
confound_cols = [
    "trans_x", "trans_y", "trans_z",
    "rot_x", "rot_y", "rot_z",
    "framewise_displacement"
]
confounds = all_confounds[confound_cols].fillna(0)

print("Loading events/regressors...")
events = pd.read_table(events_path)

# ========= 3. BUILD DESIGN MATRIX =========

# Get TR from fMRIPrep JSON or set manually once you know it
TR = 2.0  # TODO: replace with actual TR from your data

n_scans = fmri_img.shape[-1]
frame_times = np.arange(n_scans) * TR

# Assume events.tsv has something like:
# onset (s), duration (s), trial_type ('music' vs 'rest'), arousal (z-scored)
# If not z-scored yet, you can z-score here:
if 'arousal' in events.columns:
    events['arousal'] = (events['arousal'] - events['arousal'].mean()) / events['arousal'].std()

design_matrix = make_first_level_design_matrix(
    frame_times,
    events=events,
    hrf_model='glover',
    add_regs=None,          # extra regressors if needed
    add_reg_names=None
)

print("Design matrix columns:", design_matrix.columns)

# ========= 4. FIT FIRST-LEVEL GLM =========

first_level_model = FirstLevelModel(
    t_r=TR,
    slice_time_ref=0.5,
    mask_img=mask_img,
    hrf_model='glover',
    drift_model='cosine',
    high_pass=1/128,          # ~128s cutoff
)

print("Fitting GLM...")
first_level_model = first_level_model.fit(
    fmri_img,
    design_matrices=design_matrix,
    confounds=confounds
)

# ========= 5. DEFINE CONTRASTS =========

# Example: music > rest (if trial_type coding supports it)
# design_matrix might have columns like: 'music', 'rest', 'arousal'
contrast_music_vs_rest = np.array(
    [1 if name == 'music' else -1 if name == 'rest' else 0
     for name in design_matrix.columns]
)

# Parametric effect of arousal
contrast_arousal = np.array(
    [1 if name == 'arousal' else 0 for name in design_matrix.columns]
)

# ========= 6. COMPUTE & PLOT Z-MAPS =========

print("Computing contrasts...")

z_map_music_rest = first_level_model.compute_contrast(
    contrast_music_vs_rest,
    output_type='z_score'
)

z_map_arousal = first_level_model.compute_contrast(
    contrast_arousal,
    output_type='z_score'
)

out_dir = os.path.join(PROJECT_ROOT, "glm_results", SUBJECT, RUN)
os.makedirs(out_dir, exist_ok=True)

music_rest_path = os.path.join(out_dir, "zmap_music_vs_rest.nii.gz")
arousal_path = os.path.join(out_dir, "zmap_arousal.nii.gz")

z_map_music_rest.to_filename(music_rest_path)
z_map_arousal.to_filename(arousal_path)

print("Saved:", music_rest_path)
print("Saved:", arousal_path)

print("Plotting Arousal+ map...")
plotting.plot_stat_map(
    z_map_arousal,
    title=f"{SUBJECT} {RUN} Arousal+",
    threshold=3.1,  # ~p<0.001 uncorrected
    display_mode='z',
    cut_coords=7
)
plotting.show()

How to use this once data are ready

1. When the Data & Preprocessing + Behavioral folks are done, you:
2. Fix PROJECT_ROOT
3. Fix the TR
4. Make sure the filenames in the os.path.join(...) bits match your actual fMRIPrep and regressor filenames
5. Run this script for one subject, one run

If it works, loop over subjects and runs later